In [10]:
#!/usr/bin/env python3
"""
Create world maps for PLURALISTIC IGNORANCE - RUN THIS LOCALLY
Requires: pip install geopandas matplotlib pandas requests

This creates maps showing pluralistic ignorance (actual norm - perceived norm).
Positive values = people underestimate others' willingness (pluralistic ignorance)
Negative values = people overestimate others' willingness

UPDATED: Using 50m (medium resolution) to capture small countries like Singapore, Malta, Mauritius
"""

import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend for better compatibility
import matplotlib.pyplot as plt
import geopandas as gpd
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
import os

# Set matplotlib parameters
matplotlib.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'font.family': 'sans-serif'
})

def download_naturalearth_data():
    """Download Natural Earth MEDIUM resolution data (50m) - includes small countries."""
    import requests
    import zipfile
    from io import BytesIO
    
    cache_dir = os.path.expanduser('~/.cache/naturalearth')
    os.makedirs(cache_dir, exist_ok=True)
    
    # UPDATED: Using 50m (medium resolution) instead of 110m to capture small countries
    shapefile_path = os.path.join(cache_dir, 'ne_50m_admin_0_countries.shp')
    
    if os.path.exists(shapefile_path):
        print(f'Using cached shapefile: {shapefile_path}')
        return shapefile_path
    
    print('Downloading Natural Earth 50m (medium resolution) data...')
    url = 'https://naciscdn.org/naturalearth/50m/cultural/ne_50m_admin_0_countries.zip'
    
    response = requests.get(url)
    response.raise_for_status()
    
    with zipfile.ZipFile(BytesIO(response.content)) as z:
        z.extractall(cache_dir)
    
    print(f'Downloaded to: {cache_dir}')
    return shapefile_path

def create_country_mapping():
    """Map your country names to Natural Earth names."""
    return {
        'Bosnia Herzegovina': 'Bosnia and Herz.',
        'Central African Republic': 'Central African Rep.',
        'Congo Brazzaville': 'Congo',
        'Czech Republic': 'Czechia',
        'Democratic Republic of Congo': 'Dem. Rep. Congo',
        'Dominican Republic': 'Dominican Rep.',
        'Equatorial Guinea': 'Eq. Guinea',
        'Hong Kong': 'Hong Kong',
        'Ivory Coast': "Côte d'Ivoire",
        'Kyrgyz Republic': 'Kyrgyzstan',
        'Laos': 'Laos',
        'Macedonia': 'North Macedonia',
        'Malta': 'Malta',
        'Mauritius': 'Mauritius',
        'Republic of Congo': 'Congo',
        'Singapore': 'Singapore',
        'Slovakia': 'Slovakia',
        'Solomon Islands': 'Solomon Is.',
        'South Korea': 'South Korea',
        'South Sudan': 'S. Sudan',
        'Timor Leste': 'Timor-Leste',
        'United Kingdom': 'United Kingdom',
        'United States': 'United States of America',
        'West Bank and Gaza': 'Palestine',
    }

def plot_single_map(world_data, column, title, output_file, vmin=-30, vmax=30, is_pi=True):
    """Create a single world map - FIXED for matplotlib 3.10+
    
    Args:
        is_pi: If True, uses diverging colormap centered at 0 for pluralistic ignorance
               If False, uses sequential colormap (for perceived norms)
    """
    fig, ax = plt.subplots(1, 1, figsize=(14, 7), dpi=300)
    
    if is_pi:
        # Diverging colormap for pluralistic ignorance (red = positive PI, blue = negative PI)
        cmap = matplotlib.colormaps['RdBu_r']  # Red for positive (underestimate), Blue for negative (overestimate)
        label_text = 'Pluralistic Ignorance (pp)'
    else:
        # Sequential colormap for perceived norms
        colors = ['#1a9850', '#91cf60', '#d9ef8b', '#fee08b', '#fc8d59', '#d73027']
        cmap = LinearSegmentedColormap.from_list('custom', colors, N=100)
        label_text = 'Belief about others\' willingness (%)'
    
    world_data.plot(
        column=column,
        ax=ax,
        legend=True,
        cmap=cmap,
        edgecolor='white',
        linewidth=0.5,
        missing_kwds={'color': 'lightgrey'},
        vmin=vmin,
        vmax=vmax,
        legend_kwds={
            'label': label_text,
            'orientation': 'horizontal',
            'shrink': 0.5,
            'pad': 0.05
        }
    )
    
    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
    ax.axis('off')
    
    # Add statistics
    data = world_data[column].dropna()
    mean_val, median_val = data.mean(), data.median()
    if is_pi:
        textstr = f'Mean: {mean_val:.1f}pp\nMedian: {median_val:.1f}pp\nN = {len(data)}'
    else:
        textstr = f'Mean: {mean_val:.1f}%\nMedian: {median_val:.1f}%\nN = {len(data)}'
    ax.text(0.02, 0.98, textstr, transform=ax.transAxes, 
            fontsize=10, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
    
    plt.tight_layout()
    plt.savefig(f'{output_file}.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'{output_file}.pdf', bbox_inches='tight')
    plt.close()
    print(f'✓ Saved {output_file}')

def plot_comparison_grid_5panel(world_data, output_file='llm_comparison_pi'):
    """Create 2x3 comparison grid showing pluralistic ignorance (5 panels + 1 empty) - FIXED for matplotlib 3.10+"""
    fig, axes = plt.subplots(2, 3, figsize=(18, 10), dpi=300)
    axes = axes.flatten()
    
    panels = [
        ('pi_actual', 'Ground Truth PI'),
        ('pi_claude', 'Claude PI'),
        ('pi_llama', 'Llama PI'),
        ('pi_gpt', 'GPT-4 PI'),
        ('pi_gemini', 'Gemini PI'),
    ]
    
    # Diverging colormap for pluralistic ignorance
    cmap = matplotlib.colormaps['RdBu_r']  # Red = positive PI (underestimate), Blue = negative PI (overestimate)
    
    for idx, (column, title) in enumerate(panels):
        ax = axes[idx]
        
        world_data.plot(
            column=column,
            ax=ax,
            cmap=cmap,
            edgecolor='white',
            linewidth=0.3,
            vmin=-30,
            vmax=30,
            legend=False,
            missing_kwds={'color': 'lightgrey'}
        )
        
        letter = chr(65 + idx)
        ax.set_title(f'{letter}. {title}', fontsize=12, fontweight='bold', loc='left')
        ax.axis('off')
        
        mean_val = world_data[column].dropna().mean()
        ax.text(0.02, 0.98, f'Mean: {mean_val:.1f}pp', transform=ax.transAxes,
                fontsize=9, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
    
    # Hide the 6th subplot
    axes[5].axis('off')
    
    # Add legend in the empty space
    legend_text = """
Pluralistic Ignorance:
Actual norm - Perceived norm

Red (Positive): People underestimate 
others' willingness (classic PI)

Blue (Negative): People overestimate 
others' willingness

Ground Truth: Survey data
LLMs: Model predictions
"""
    axes[5].text(0.5, 0.5, legend_text, transform=axes[5].transAxes,
                 fontsize=11, ha='center', va='center',
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    
    # Shared colorbar
    from matplotlib.cm import ScalarMappable
    cax = fig.add_axes([0.92, 0.3, 0.015, 0.4])
    sm = ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=-30, vmax=30))
    sm.set_array([])
    cbar = plt.colorbar(sm, cax=cax)
    cbar.set_label('Pluralistic Ignorance (pp)', rotation=270, labelpad=20)
    
    plt.suptitle('Pluralistic Ignorance in Climate Beliefs: Ground Truth vs. LLM Predictions', 
                 fontsize=14, fontweight='bold', y=0.98)
    plt.tight_layout(rect=[0, 0, 0.91, 0.96])
    
    plt.savefig(f'{output_file}.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'{output_file}.pdf', bbox_inches='tight')
    plt.close()
    print(f'✓ Saved {output_file}')

def plot_comparison_grid_3rows(world_data, output_file='llm_comparison_pi_3rows'):
    """Create 3-row layout: Ground truth centered on top, 4 LLMs in 2x2 below - FIXED for matplotlib 3.10+"""
    fig = plt.figure(figsize=(18, 16), dpi=300)
    
    # Diverging colormap for pluralistic ignorance
    cmap = matplotlib.colormaps['RdBu_r']  # Red = positive PI (underestimate), Blue = negative PI (overestimate)
    
    # Top row: Ground truth (centered, spanning 2 columns)
    ax_top = plt.subplot2grid((3, 2), (0, 0), colspan=2, fig=fig)
    
    world_data.plot(
        column='pi_actual',
        ax=ax_top,
        cmap=cmap,
        edgecolor='white',
        linewidth=0.3,
        vmin=-30,
        vmax=30,
        legend=False,
        missing_kwds={'color': 'lightgrey'}
    )
    
    ax_top.set_title('A. Ground Truth: Pluralistic Ignorance', fontsize=14, fontweight='bold', loc='left', pad=15)
    ax_top.axis('off')
    
    mean_val = world_data['pi_actual'].dropna().mean()
    ax_top.text(0.02, 0.98, f'Mean: {mean_val:.1f}pp', transform=ax_top.transAxes,
                fontsize=11, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
    
    # Middle and bottom rows: 4 LLMs in 2x2 grid
    llm_panels = [
        ('pi_claude', 'Claude PI', (1, 0)),
        ('pi_llama', 'Llama PI', (1, 1)),
        ('pi_gpt', 'GPT-4 PI', (2, 0)),
        ('pi_gemini', 'Gemini PI', (2, 1)),
    ]
    
    for idx, (column, title, position) in enumerate(llm_panels):
        ax = plt.subplot2grid((3, 2), position, fig=fig)
        
        world_data.plot(
            column=column,
            ax=ax,
            cmap=cmap,
            edgecolor='white',
            linewidth=0.3,
            vmin=-30,
            vmax=30,
            legend=False,
            missing_kwds={'color': 'lightgrey'}
        )
        
        letter = chr(66 + idx)  # Start from 'B' since 'A' is ground truth
        ax.set_title(f'{letter}. {title}', fontsize=12, fontweight='bold', loc='left')
        ax.axis('off')
        
        mean_val = world_data[column].dropna().mean()
        ax.text(0.02, 0.98, f'Mean: {mean_val:.1f}pp', transform=ax.transAxes,
                fontsize=9, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
    
    # Add shared colorbar on the right
    from matplotlib.cm import ScalarMappable
    cax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
    sm = ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=-30, vmax=30))
    sm.set_array([])
    cbar = plt.colorbar(sm, cax=cax)
    cbar.set_label('Pluralistic Ignorance (pp)', rotation=270, labelpad=20, fontsize=11)
    
    plt.suptitle('Pluralistic Ignorance in Climate Beliefs: Ground Truth vs. LLM Predictions', 
                 fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout(rect=[0, 0, 0.91, 0.97])
    
    plt.savefig(f'{output_file}.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'{output_file}.pdf', bbox_inches='tight')
    plt.close()
    print(f'✓ Saved {output_file}')

def plot_error_maps(world_data, output_file='llm_pi_error_comparison'):
    """Create comparison of PI prediction errors (MAE from ground truth PI) - FIXED for matplotlib 3.10+"""
    fig, axes = plt.subplots(2, 2, figsize=(16, 10), dpi=300)
    axes = axes.flatten()
    
    # Calculate PI errors (absolute difference from ground truth PI)
    world_data['error_pi_claude'] = abs(world_data['pi_claude'] - world_data['pi_actual'])
    world_data['error_pi_llama'] = abs(world_data['pi_llama'] - world_data['pi_actual'])
    world_data['error_pi_gpt'] = abs(world_data['pi_gpt'] - world_data['pi_actual'])
    world_data['error_pi_gemini'] = abs(world_data['pi_gemini'] - world_data['pi_actual'])
    
    models = [
        ('error_pi_claude', 'Claude PI Error'),
        ('error_pi_llama', 'Llama PI Error'),
        ('error_pi_gpt', 'GPT-4 PI Error'),
        ('error_pi_gemini', 'Gemini PI Error')
    ]
    
    # Red colormap (more red = more error)
    cmap = matplotlib.colormaps['Reds']
    
    for idx, (column, title) in enumerate(models):
        ax = axes[idx]
        
        world_data.plot(
            column=column,
            ax=ax,
            cmap=cmap,
            edgecolor='white',
            linewidth=0.3,
            vmin=0,
            vmax=25,
            legend=False,
            missing_kwds={'color': 'lightgrey'}
        )
        
        letter = chr(65 + idx)
        ax.set_title(f'{letter}. {title}', fontsize=12, fontweight='bold', loc='left')
        ax.axis('off')
        
        mae = world_data[column].dropna().mean()
        ax.text(0.02, 0.98, f'MAE: {mae:.2f}pp', transform=ax.transAxes,
                fontsize=9, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
    
    # Shared colorbar
    from matplotlib.cm import ScalarMappable
    cax = fig.add_axes([0.92, 0.3, 0.015, 0.4])
    sm = ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=0, vmax=25))
    sm.set_array([])
    cbar = plt.colorbar(sm, cax=cax)
    cbar.set_label('PI Prediction Error (pp)', rotation=270, labelpad=20)
    
    plt.suptitle('LLM Pluralistic Ignorance Prediction Errors by Country', 
                 fontsize=14, fontweight='bold', y=0.98)
    plt.tight_layout(rect=[0, 0, 0.91, 0.96])
    
    plt.savefig(f'{output_file}.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'{output_file}.pdf', bbox_inches='tight')
    plt.close()
    print(f'✓ Saved {output_file}')

def main():
    """Main function - UPDATE THE CSV_PATH TO YOUR LOCAL PATH"""
    print('='*80)
    print('CREATING WORLD MAPS - PLURALISTIC IGNORANCE')
    print('Using 50m (medium resolution) to capture small countries')
    print('='*80)
    
    print('\nLoading data...')
    # UPDATE THIS PATH TO WHERE YOU SAVED THE FILE
    csv_path = 'comprehensive_data_for_mapping.csv'
    df = pd.read_csv(csv_path)
    
    print(f'✓ Loaded data with {len(df)} rows')
    print(f'\nAvailable columns:')
    for col in sorted(df.columns):
        print(f'  - {col}')
    
    print('\nLoading world geometries...')
    # Download Natural Earth data (50m resolution)
    shapefile_path = download_naturalearth_data()
    world = gpd.read_file(shapefile_path)
    
    # FIXED: Check what column name exists in the shapefile
    print(f'Available columns in world shapefile: {list(world.columns)}')
    
    # Determine the correct column name to use for merging
    if 'NAME' in world.columns:
        merge_column = 'NAME'
    elif 'ADMIN' in world.columns:
        merge_column = 'ADMIN'
    elif 'name' in world.columns:
        merge_column = 'name'
    else:
        # Default to first string column
        merge_column = world.select_dtypes(include=['object']).columns[0]
    
    print(f'Using column "{merge_column}" for country matching')
    
    # Apply country mapping
    country_map = create_country_mapping()
    df['country_mapped'] = df['countrynew'].replace(country_map)
    
    print('\nCalculating pluralistic ignorance...')
    print('PI = Actual norm (own willingness) - Perceived norm (belief about others)')
    
    # Check if PI is already calculated
    if 'pi_actual' in df.columns:
        print('✓ Found pre-calculated PI values in data!')
        print('  Using existing columns: pi_actual, pi_pred_claude, pi_pred_llama, pi_pred_gpt, pi_pred_gemini')
        
        # Rename prediction columns to match our naming convention
        df['pi_claude'] = df['pi_pred_claude']
        df['pi_llama'] = df['pi_pred_llama']
        df['pi_gpt'] = df['pi_pred_gpt']
        df['pi_gemini'] = df['pi_pred_gemini']
    else:
        # Calculate PI from scratch
        print('Calculating PI from raw data...')
        
        # Check for own_willingness column (try different possible names)
        own_will_col = None
        possible_names = ['own_willingness_pct', 'own_willingness_actual_pct', 'own_willingness', 'actual_own_willingness']
        for name in possible_names:
            if name in df.columns:
                own_will_col = name
                print(f'✓ Found own willingness column: {name}')
                break
        
        if own_will_col is None:
            print('\n❌ ERROR: Could not find own_willingness column!')
            print('Available columns containing "willing":', [c for c in df.columns if 'willing' in c.lower()])
            print('\nPlease check your data file and update the column name in the script.')
            return
        
        # Calculate pluralistic ignorance for ground truth
        df['pi_actual'] = df[own_will_col] - df['other_willingness_actual_pct']
        
        # Calculate pluralistic ignorance for LLM predictions
        df['pi_claude'] = df[own_will_col] - df['pred_claude']
        df['pi_llama'] = df[own_will_col] - df['pred_llama']
        df['pi_gpt'] = df[own_will_col] - df['pred_gpt']
        df['pi_gemini'] = df[own_will_col] - df['pred_gemini']
    
    print(f'✓ Calculated PI for ground truth and all LLM predictions')
    print(f'  Ground truth PI mean: {df["pi_actual"].mean():.2f}pp')
    print(f'  Claude PI mean: {df["pi_claude"].mean():.2f}pp')
    print(f'  Llama PI mean: {df["pi_llama"].mean():.2f}pp')
    print(f'  GPT PI mean: {df["pi_gpt"].mean():.2f}pp')
    print(f'  Gemini PI mean: {df["pi_gemini"].mean():.2f}pp')
    
    # Merge using the correct column
    world_data = world.merge(df, left_on=merge_column, right_on='country_mapped', how='left')
    
    matched = world_data['pi_actual'].notna().sum()
    total = len(df)
    print(f'\nMatched {matched}/{total} countries')
    
    if matched < total:
        missing = set(df['countrynew']) - set(world_data[world_data['pi_actual'].notna()]['countrynew'])
        print(f'Missing countries: {", ".join(sorted(missing))}')
    
    print('='*80)
    print('Creating individual PI maps...')
    print('='*80 + '\n')
    
    # Ground truth PI
    plot_single_map(world_data, 'pi_actual', 
                   'Ground Truth: Pluralistic Ignorance', 
                   'map_pi_ground_truth',
                   vmin=-30, vmax=30, is_pi=True)
    
    # Individual model PI predictions
    plot_single_map(world_data, 'pi_claude', 'Claude - Predicted Pluralistic Ignorance', 
                   'map_pi_claude', vmin=-30, vmax=30, is_pi=True)
    plot_single_map(world_data, 'pi_llama', 'Llama - Predicted Pluralistic Ignorance', 
                   'map_pi_llama', vmin=-30, vmax=30, is_pi=True)
    plot_single_map(world_data, 'pi_gpt', 'GPT-4 - Predicted Pluralistic Ignorance', 
                   'map_pi_gpt', vmin=-30, vmax=30, is_pi=True)
    plot_single_map(world_data, 'pi_gemini', 'Gemini - Predicted Pluralistic Ignorance', 
                   'map_pi_gemini', vmin=-30, vmax=30, is_pi=True)
    
    print('\n' + '='*80)
    print('Creating comparison grids...')
    print('='*80 + '\n')
    
    # Comparison grid with ground truth PI (2x3 layout)
    plot_comparison_grid_5panel(world_data)
    
    # 3-row comparison grid (Ground truth centered on top)
    plot_comparison_grid_3rows(world_data)
    
    # PI Error maps
    plot_error_maps(world_data)
    
    print('\n' + '='*80)
    print('✓ ALL PLURALISTIC IGNORANCE MAPS CREATED SUCCESSFULLY!')
    print('='*80)
    print('\nFiles created:')
    print('  Individual PI maps:')
    print('    - map_pi_ground_truth.png/.pdf')
    print('    - map_pi_claude.png/.pdf')
    print('    - map_pi_llama.png/.pdf')
    print('    - map_pi_gpt.png/.pdf')
    print('    - map_pi_gemini.png/.pdf')
    print('  Comparison grids:')
    print('    - llm_comparison_pi.png/.pdf (2x3 layout: 5 panels + legend)')
    print('    - llm_comparison_pi_3rows.png/.pdf (3-row layout: GT centered on top)')
    print('    - llm_pi_error_comparison.png/.pdf (PI error maps)')
    print('\nInterpretation:')
    print('  Red (positive values): People underestimate others\' willingness (classic PI)')
    print('  Blue (negative values): People overestimate others\' willingness')
    print('  White (zero): Accurate perception of others\' willingness')

if __name__ == '__main__':
    main()

CREATING WORLD MAPS - PLURALISTIC IGNORANCE
Using 50m (medium resolution) to capture small countries

Loading data...
✓ Loaded data with 125 rows

Available columns:
  - country
  - countrynew
  - gdp_capita_2021
  - hdi_2021
  - other_willingness_actual_pct
  - own_willingness_pct
  - pi_actual
  - pi_pred_claude
  - pi_pred_gemini
  - pi_pred_gpt
  - pi_pred_llama
  - pred_claude
  - pred_gemini
  - pred_gpt
  - pred_llama
  - temp_mean_2010_2019

Loading world geometries...
Using cached shapefile: /root/.cache/naturalearth/ne_50m_admin_0_countries.shp
Available columns in world shapefile: ['featurecla', 'scalerank', 'LABELRANK', 'SOVEREIGNT', 'SOV_A3', 'ADM0_DIF', 'LEVEL', 'TYPE', 'TLC', 'ADMIN', 'ADM0_A3', 'GEOU_DIF', 'GEOUNIT', 'GU_A3', 'SU_DIF', 'SUBUNIT', 'SU_A3', 'BRK_DIFF', 'NAME', 'NAME_LONG', 'BRK_A3', 'BRK_NAME', 'BRK_GROUP', 'ABBREV', 'POSTAL', 'FORMAL_EN', 'FORMAL_FR', 'NAME_CIAWF', 'NOTE_ADM0', 'NOTE_BRK', 'NAME_SORT', 'NAME_ALT', 'MAPCOLOR7', 'MAPCOLOR8', 'MAPCOLOR9', '

In [4]:
#!/usr/bin/env python3
"""
DIAGNOSTIC: Find the 3 remaining unmatched countries
"""

import pandas as pd
import geopandas as gpd
import os

def download_naturalearth_data():
    import requests
    import zipfile
    from io import BytesIO
    
    cache_dir = os.path.expanduser('~/.cache/naturalearth')
    os.makedirs(cache_dir, exist_ok=True)
    
    shapefile_path = os.path.join(cache_dir, 'ne_50m_admin_0_countries.shp')
    
    if os.path.exists(shapefile_path):
        return shapefile_path
    
    print('Downloading Natural Earth 50m data...')
    url = 'https://naciscdn.org/naturalearth/50m/cultural/ne_50m_admin_0_countries.zip'
    
    response = requests.get(url)
    response.raise_for_status()
    
    with zipfile.ZipFile(BytesIO(response.content)) as z:
        z.extractall(cache_dir)
    
    return shapefile_path

def create_country_mapping():
    return {
        'Bosnia Herzegovina': 'Bosnia and Herz.',
        'Central African Republic': 'Central African Rep.',
        'Congo Brazzaville': 'Congo',
        'Czech Republic': 'Czechia',
        'Democratic Republic of Congo': 'Dem. Rep. Congo',
        'Dominican Republic': 'Dominican Rep.',
        'Equatorial Guinea': 'Eq. Guinea',
        'Hong Kong': 'Hong Kong S.A.R.',
        'Ivory Coast': "Côte d'Ivoire",
        'Kyrgyz Republic': 'Kyrgyzstan',
        'Laos': 'Lao PDR',
        'Macedonia': 'North Macedonia',
        'Malta': 'Malta',
        'Mauritius': 'Mauritius',
        'Republic of Congo': 'Congo',
        'Singapore': 'Singapore',
        'Slovakia': 'Slovakia',
        'Solomon Islands': 'Solomon Is.',
        'South Korea': 'Korea',
        'South Sudan': 'S. Sudan',
        'Timor Leste': 'Timor-Leste',
        'United Kingdom': 'United Kingdom',
        'United States': 'United States of America',
        'West Bank and Gaza': 'Palestine',
    }

# Load data
csv_path = 'comprehensive_data_for_mapping.csv'
df = pd.read_csv(csv_path)

# Load Natural Earth 50m
shapefile_path = download_naturalearth_data()
world = gpd.read_file(shapefile_path)
merge_column = 'NAME'

# Get Natural Earth country names
ne_countries = set(world[merge_column].dropna().unique())

# Apply mapping
country_map = create_country_mapping()
df['country_mapped'] = df['countrynew'].replace(country_map)

# Find unmatched
unmatched = []
matched = []

for country in sorted(df['countrynew'].unique()):
    mapped_name = country_map.get(country, country)
    if mapped_name in ne_countries:
        matched.append((country, mapped_name))
    else:
        unmatched.append((country, mapped_name))

print("="*80)
print(f"RESULTS: {len(matched)}/125 matched, {len(unmatched)} unmatched")
print("="*80)

if unmatched:
    print("\nTHE 3 MISSING COUNTRIES:")
    print("-" * 80)
    for original, mapped in unmatched:
        print(f"\n  {original} -> {mapped}")
        
        # Try to find similar names
        search_words = set(original.lower().split())
        candidates = []
        for ne_country in ne_countries:
            ne_words = set(ne_country.lower().split())
            # Look for any matching words
            if search_words & ne_words:
                candidates.append(ne_country)
        
        if candidates:
            print(f"  Possible matches in shapefile:")
            for candidate in sorted(candidates)[:5]:
                print(f"    - {candidate}")
        else:
            print(f"  No obvious matches found")

print("\n" + "="*80)

RESULTS: 122/125 matched, 3 unmatched

THE 3 MISSING COUNTRIES:
--------------------------------------------------------------------------------

  Hong Kong -> Hong Kong S.A.R.
  Possible matches in shapefile:
    - Hong Kong

  Laos -> Lao PDR
  Possible matches in shapefile:
    - Laos

  South Korea -> Korea
  Possible matches in shapefile:
    - North Korea
    - South Africa
    - South Korea



<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=feb9f195-de2a-416f-b8f1-09efca4e954f' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>